# IEEE-CIS Fraud Risk Data Pipeline

This notebook is the educational companion to the executable Spark pipeline. It documents ingestion, quality controls, leakage-safe temporal splitting, feature engineering, imbalance variants, and the Docker handoff. The full run is opt-in because the source data is larger than 1 GB.

## tl;dr

Run the Docker Compose workflow to create reusable Parquet datasets under `data/processed/ieee_cis_fraud_risk`. Downstream users should start with `model_ready/train_weighted`.

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config' / 'pipeline_config.yaml').exists():
    PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'data' / 'ieee-fraud-detection'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ieee_cis_fraud_risk'
REQUIRED = ['train_transaction.csv', 'train_identity.csv', 'test_transaction.csv', 'test_identity.csv']
inventory = {name: (RAW_DIR / name).stat().st_size for name in REQUIRED if (RAW_DIR / name).exists()}
print({'raw_dir': str(RAW_DIR), 'files_found': list(inventory), 'total_mb': round(sum(inventory.values()) / 1024**2, 2)})

## Context & Methods

Spark DataFrames are the primary engine. Identity tables are normalized and left-joined by `TransactionID`; the labeled data is split chronologically 70/15/15 using `TransactionDT`. Numeric medians, category handling, and entity aggregates are fitted on training data only. Validation and holdout remain untouched.

### Key assumptions

`TransactionDT` is relative time, not a calendar timestamp. Fraud rates are evaluated with PR-AUC, recall, precision, F1, and ROC-AUC; accuracy is secondary because of class imbalance.

## Run the pipeline

From the repository root:

```powershell
docker compose -f docker-compose.preprocessing.yml build
docker compose -f docker-compose.preprocessing.yml run --rm preprocess
docker compose -f docker-compose.preprocessing.yml run --rm verify-processed
```

In [ ]:
# Optional local smoke check; the full Spark job is intentionally opt-in.
RUN_FULL_PIPELINE = False
if RUN_FULL_PIPELINE:
    from pipeline.fraud_risk_data_pipeline import main
    main([])
else:
    print('Full pipeline not started. Use Docker Compose commands above.')

## Data and outputs

The output contract contains curated joined/feature data, chronological splits, training variants (`train_original`, `train_weighted`, `train_balanced`), feature-store aggregates, reports, model artifacts, demo cases, and `manifest.json`. The weighted training set is recommended for an imbalance-aware baseline; validation and holdout are not resampled.

## Checks

The verifier checks the manifest, required datasets, and non-empty output paths. Runtime completion must be confirmed by the verifier; this notebook does not claim results before the pipeline has executed.